In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True


In [2]:
sys.path.append('/content/drive/MyDrive')

In [3]:
"""
t8_cpu_latency.py  —  T8 (review item M3)

The paper measures latency only on a Tesla T4 GPU, which a reviewer can
read as not representative of ARM-class OBU/RSU hardware. This adds a
CPU-ONLY latency profile (the closest proxy available without the actual
embedded board) for the two configurations that matter:
    XGBoost-only  and  BiLSTM-15s.

It does NOT claim to be OBU hardware -- it is a "reference CPU" point that
makes the latency story less GPU-bound. Methodology mirrors the paper:
100 untimed warm-up passes, then 1000 timed single-sample forward passes;
report mean, std, p50, p95, p99.

GPU is disabled BEFORE importing TensorFlow so the run is genuinely CPU-only.
NEEDS THE DATASET. One seed is enough for a latency profile (weights do not
affect forward-pass timing materially); set SEED below.
"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''   # force CPU — must precede TF import
import json, time
import numpy as np
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from common import (load_dataset, split_by_episode, create_windows,
                    pad_and_normalize, extract_tabular_features,
                    set_all_seeds, WINDOW_SEC)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Dropout,
                                     BatchNormalization, Bidirectional)

In [5]:
SEED = 42
N_WARMUP = 100
N_TIMED  = 1000

In [6]:
def build_bilstm15(input_shape, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    inp = Input(shape=input_shape)
    x = Bidirectional(LSTM(128, return_sequences=True))(inp)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    emb = Dense(32, activation='relu')(x)
    out = Dense(3, activation='softmax')(emb)
    return Model(inp, out)

In [7]:
def percentiles(times_ms):
    a = np.array(times_ms)
    return dict(mean=float(a.mean()), std=float(a.std(ddof=1)),
                p50=float(np.percentile(a, 50)),
                p95=float(np.percentile(a, 95)),
                p99=float(np.percentile(a, 99)),
                min=float(a.min()), max=float(a.max()))

In [8]:
def time_xgb(df_speed):
    set_all_seeds(SEED)
    tr, va, te = split_by_episode(df_speed, seed=SEED)
    Xtr_l, ytr, _ = create_windows(tr, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(va, WINDOW_SEC)
    Xte_l, yte, _ = create_windows(te, WINDOW_SEC)
    Xtr_s, Xv_s, Xte_s, _ = pad_and_normalize(Xtr_l, Xv_l, Xte_l)
    Xtr = extract_tabular_features(Xtr_s)
    Xv  = extract_tabular_features(Xv_s)
    Xte = extract_tabular_features(Xte_s)
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xv = sc.transform(Xv); Xte = sc.transform(Xte)
    xgb = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=200,
                        max_depth=5, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, random_state=SEED,
                        eval_metric='mlogloss', early_stopping_rounds=20,
                        n_jobs=1)            # single thread = OBU-like
    xgb.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
    one = Xte[:1]
    for _ in range(N_WARMUP):
        xgb.predict(one)
    times = []
    for _ in range(N_TIMED):
        t = time.perf_counter(); xgb.predict(one); times.append((time.perf_counter()-t)*1000)
    return percentiles(times)

In [9]:
def time_bilstm(df_speed):
    set_all_seeds(SEED)
    tr, va, te = split_by_episode(df_speed, seed=SEED)
    Xtr_l, ytr, _ = create_windows(tr, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(va, WINDOW_SEC)
    Xte_l, yte, _ = create_windows(te, WINDOW_SEC)
    Xtr_s, Xv_s, Xte_s, mlen = pad_and_normalize(Xtr_l, Xv_l, Xte_l)
    model = build_bilstm15((Xtr_s.shape[1], Xtr_s.shape[2]))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    model.fit(Xtr_s, ytr, validation_data=(Xv_s, yv), epochs=5,
              batch_size=32, verbose=0)    # brief fit; timing is weight-agnostic
    one = Xte_s[:1]
    for _ in range(N_WARMUP):
        model.predict(one, verbose=0)
    times = []
    for _ in range(N_TIMED):
        t = time.perf_counter(); model.predict(one, verbose=0)
        times.append((time.perf_counter()-t)*1000)
    return percentiles(times)

In [10]:
def main(outdir='outputs_cpu_latency'):
    os.makedirs(outdir, exist_ok=True)
    print(f'TensorFlow GPUs visible: {tf.config.list_physical_devices("GPU")} '
          f'(should be empty = CPU-only)')
    results = {}
    for spd in (25, 15):
        df = load_dataset(spd)
        results[f'{spd}ms'] = {
            'xgboost_only_cpu_ms': time_xgb(df),
            'bilstm15_cpu_ms':     time_bilstm(df),
        }
        r = results[f'{spd}ms']
        print('\n' + '=' * 70)
        print(f'  T8 — CPU-only latency @ {spd} m/s (ms, single sample)')
        print('=' * 70)
        for name, key in [('XGBoost-only', 'xgboost_only_cpu_ms'),
                          ('BiLSTM-15s', 'bilstm15_cpu_ms')]:
            d = r[key]
            print(f'  {name:<14s} mean {d["mean"]:.3f} ± {d["std"]:.3f}  '
                  f'p50 {d["p50"]:.3f}  p95 {d["p95"]:.3f}  p99 {d["p99"]:.3f}')

    with open(os.path.join(outdir, 't8_cpu_latency.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nSaved to {outdir}/')
    print('Frame this as a reference-CPU point, NOT as OBU/RSU validation.')

In [11]:
if __name__ == '__main__':
    main()

TensorFlow GPUs visible: [] (should be empty = CPU-only)

  T8 — CPU-only latency @ 25 m/s (ms, single sample)
  XGBoost-only   mean 0.982 ± 0.830  p50 0.785  p95 2.886  p99 4.733
  BiLSTM-15s     mean 95.705 ± 24.569  p50 82.925  p95 143.970  p99 156.294

  T8 — CPU-only latency @ 15 m/s (ms, single sample)
  XGBoost-only   mean 0.442 ± 0.079  p50 0.423  p95 0.525  p99 0.808
  BiLSTM-15s     mean 97.094 ± 23.054  p50 84.784  p95 141.812  p99 160.248

Saved to outputs_cpu_latency/
Frame this as a reference-CPU point, NOT as OBU/RSU validation.
